# Lab 5.4 &mdash; Human-in-the-Loop as an Orchestration Mechanism

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Interrupt a run before the irreversible node and checkpoint what it knows
- Open the gate for a named human &mdash; an identity, not a boolean
- Time out, and climb an escalation ladder that actually terminates
- Prove the one property that makes a gate a gate: at the gate, &ldquo;no&rdquo; is still free

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Module 3's checkpointing, applied.** You can only pause a run whose state you can
> write down and pick up again &mdash; an approval gate is that mechanism with a person in it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the specialists
# Deterministic stand-ins. Each takes the run state and returns a PARTIAL state -- exactly
# the LangGraph node shape from Module 3 -- and reports what it spent. No model is called,
# so a graph's structure AND its cost can be graded offline and exactly. The "Run it for
# real" cells put the sandbox model behind the same interface.

SANCTIONS_WATCH = {"NORTHWIND"}

COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}


def agent_ledger(state: dict) -> dict:
    """Read the payment named in the state."""
    ref = state.get("ref")
    record = LEDGER.get(ref)
    if record is None:
        return {"problems": [f"no payment on file with reference {ref!r}"],
                "tokens": COST["ledger"]}
    return {"facts": {"ref": ref, **record},
            "findings": [{"by": "ledger", "source": "ledger",
                          "claim": f"{ref} is {record['status']} "
                                   f"for {record['amount']:,.2f} {record['ccy']}"}],
            "tokens": COST["ledger"]}


def agent_policy(state: dict) -> dict:
    """Say what the operating policy is for whatever went wrong."""
    code = (state.get("facts") or {}).get("reason_code")
    if code is None:
        return {"problems": ["policy ran before the reason code existed"],
                "tokens": COST["policy"]}
    return {"findings": [{"by": "policy", "source": "policy",
                          "claim": POLICY.get(code, f"no policy on file for {code}")}],
            "needs_human": code in NEEDS_HUMAN,
            "tokens": COST["policy"]}


def agent_sanctions(state: dict) -> dict:
    """A set-membership test. No model needed, and none used -- note the cost column."""
    counterparty = (state.get("facts") or {}).get("counterparty")
    listed = counterparty in SANCTIONS_WATCH
    return {"findings": [{"by": "sanctions", "source": "watchlist",
                          "claim": f"{counterparty} is "
                                   f"{'ON the watchlist' if listed else 'not on the watchlist'}"}],
            "blocked": listed,
            "tokens": COST["sanctions"]}


def agent_writer(state: dict) -> dict:
    """Turn whatever findings arrived into one recommendation."""
    findings = state.get("findings") or []
    if (state.get("facts") or {}).get("status") == "settled":
        action = "no action"                      # nothing to release; it already went
    elif state.get("blocked") or state.get("needs_human"):
        action = "hold for a human"
    else:
        action = "release"
    return {"recommendation": action,
            "rationale": [f["claim"] for f in findings],
            "tokens": COST["writer"]}


AGENTS = {"ledger": agent_ledger, "policy": agent_policy,
          "sanctions": agent_sanctions, "writer": agent_writer}
print("specialists:", ", ".join(AGENTS))

## Concept

Human-in-the-loop appears twice in this course. Here it is an **orchestration mechanism**: a way
to pause a graph, ask, and carry on. In Module 8 the same machinery is a **safety control**.

Four parts, and the one people leave out is the third:

| | |
|---|---|
| **interrupt** | stop before a named node and write the state down |
| **approve** | resume, recording *who* said yes and on what evidence |
| **timeout** | a gate with no deadline is a run that waits until Monday |
| **escalate** | expiry is not refusal and not approval &mdash; it is a different queue |

## Section 1 &mdash; Interrupt before the irreversible node

The whole graph, including a `release` node that actually changes something. Stop before it.

In [ ]:
RELEASED = set()          # the irreversible side effect, so we can prove whether it happened

def node_release(state: dict) -> dict:
    """The one node that changes the world."""
    RELEASED.add(state["ref"])
    return {"released": True, "tokens": 40}


NODES = {"read": agent_ledger, "policy": agent_policy, "screen": agent_sanctions,
         "recommend": agent_writer, "release": node_release}
PLAN = ["read", "policy", "screen", "recommend", "release"]

INTERRUPT_BEFORE = {"release"}       # nodes that may not run unattended


def merge_one(state: dict, partial: dict) -> dict:
    """Fold one node's partial state in. Lists accumulate, counters add."""
    out = dict(state)
    for key, value in partial.items():
        if key == "tokens":
            out["tokens"] = out.get("tokens", 0) + value
        elif key in ("findings", "problems"):
            out[key] = (out.get(key) or []) + value
        else:
            out[key] = value
    return out


def run_until_interrupt(ref: str, plan=None, interrupt_before=None) -> dict:
    """Run nodes in order, stopping BEFORE any node that needs a human."""
    plan = PLAN if plan is None else plan
    interrupt_before = INTERRUPT_BEFORE if interrupt_before is None else interrupt_before
    state, trace = {"ref": ref, "tokens": 0}, []
    for node in plan:
        if BLANK:              # TODO: must this node wait for a human before it runs?
            return {"status": "interrupted", "at": node, "state": state,
                    "trace": trace, "remaining": plan[plan.index(node):]}
        state = merge_one(state, NODES[node](state))
        trace.append(node)
    return {"status": "completed", "at": None, "state": state, "trace": trace, "remaining": []}

In [ ]:
# --- Self-check: Section 1
def interrupted():
    RELEASED.clear()
    return run_until_interrupt("PMT-1005")

check("the run stops rather than finishing",
      lambda: interrupted()["status"] == "interrupted")
check("and it stops at the release node",
      lambda: interrupted()["at"] == "release")
check("everything before it did run",
      lambda: interrupted()["trace"] == ["read", "policy", "screen", "recommend"])
check("NOTHING WAS RELEASED",
      lambda: (interrupted(), "PMT-1005" not in RELEASED)[1] is True,
      "the point of interrupting before the node rather than after it")
check("the checkpoint carries the evidence a human needs",
      lambda: len(interrupted()["state"]["findings"]) == 3)
check("and a recommendation to agree or disagree with",
      lambda: interrupted()["state"]["recommendation"] == "hold for a human")
check("and it knows what is left to do",
      lambda: interrupted()["remaining"] == ["release"])
check("a graph with no gated node runs straight through",
      lambda: run_until_interrupt("PMT-1002", plan=["read", "policy"],
                                  interrupt_before=set())["status"] == "completed")

## Section 2 &mdash; Approval is an identity, not a boolean

Resume the checkpoint. The gate opens for a *named* person &mdash; because &ldquo;approved: true&rdquo;
answers none of the questions an auditor will ask.

In [ ]:
def resume(checkpoint: dict, approved_by=None) -> dict:
    """Resume an interrupted run. Refuse unless a named human approved it."""
    # TODO: refuse unless approved_by is a real name. A boolean is not an approver,
    # and neither is an empty string.
    if BLANK:
        return {"status": "refused", "state": checkpoint["state"],
                "reason": f"{checkpoint['at']} needs a named human approver"}
    state = dict(checkpoint["state"])
    state["approved_by"] = approved_by
    trace = list(checkpoint["trace"])
    for node in checkpoint["remaining"]:
        state = merge_one(state, NODES[node](state))
        trace.append(node)
    return {"status": "completed", "state": state, "trace": trace, "reason": ""}

In [ ]:
# --- Self-check: Section 2
def _resume(approver):
    RELEASED.clear()
    return resume(run_until_interrupt("PMT-1005"), approved_by=approver)

check("resuming with no approver is refused",
      lambda: _resume(None)["status"] == "refused")
check("and still nothing was released",
      lambda: (_resume(None), "PMT-1005" not in RELEASED)[1] is True)
check("a bare True is NOT an approver",
      lambda: _resume(True)["status"] == "refused",
      "'approved: true' cannot answer 'who approved this, and on what evidence?'")
check("nor is an empty string",
      lambda: _resume("   ")["status"] == "refused")
check("a named human opens the gate",
      lambda: _resume("ops-duty-manager")["status"] == "completed")
check("and the release actually happened",
      lambda: (_resume("ops-duty-manager"), "PMT-1005" in RELEASED)[1] is True)
check("the approver's name is on the final state",
      lambda: _resume("ops-duty-manager")["state"]["approved_by"] == "ops-duty-manager")
check("the trace shows the whole run, both halves",
      lambda: _resume("ops-duty-manager")["trace"]
              == ["read", "policy", "screen", "recommend", "release"])

## Section 3 &mdash; Timeout, and a ladder that ends

A gate with no deadline is a run that waits for someone who has gone home. Expiry is not refusal
and it is not approval &mdash; it is a different queue, and the queue eventually runs out.

In [ ]:
ESCALATION = ["ops-duty-manager", "treasury-lead", "head-of-operations"]

def escalate(current, ladder=None):
    """Who to ask next. None means the ladder is exhausted and a person must own it manually."""
    ladder = ESCALATION if ladder is None else ladder
    if current is None:
        return ladder[0]
    if current not in ladder:
        return None
    i = ladder.index(current)
    return BLANK               # TODO: the next rung up, or None if this is already the top


def gate_status(waited_s: int, deadline_s: int, approver=None) -> str:
    """What to do with a gate that has been waiting: approved, waiting, or time to escalate."""
    if isinstance(approver, str) and approver.strip():
        return "approved"
    return "waiting" if waited_s < deadline_s else "escalate"

In [ ]:
# --- Self-check: Section 3
check("an unopened gate starts at the bottom of the ladder",
      lambda: escalate(None) == "ops-duty-manager")
check("and climbs one rung at a time",
      lambda: escalate("ops-duty-manager") == "treasury-lead")
check("the ladder TERMINATES",
      lambda: escalate("head-of-operations") is None,
      "an escalation path that loops is a gate that never resolves")
check("someone outside the ladder cannot be escalated from",
      lambda: escalate("a-passing-colleague") is None)
check("inside the deadline the gate simply waits",
      lambda: gate_status(waited_s=30, deadline_s=900) == "waiting")
check("past the deadline it escalates",
      lambda: gate_status(waited_s=901, deadline_s=900) == "escalate")
check("an approval short-circuits the deadline entirely",
      lambda: gate_status(waited_s=99999, deadline_s=900, approver="treasury-lead") == "approved")
check("expiry is neither approval nor refusal",
      lambda: gate_status(waited_s=901, deadline_s=900) not in ("approved", "refused"),
      "a timeout that auto-approves is not a gate; one that auto-refuses loses real work")

def _ladder():
    who, waited = None, 0
    while True:
        who = escalate(who)
        if who is None:
            print("  ladder exhausted -- this case now belongs to a person, not to the graph")
            break
        waited += 900
        print(f"  after {waited // 60:>3} min -> ask {who}")
guard(_ladder)

## Section 4 &mdash; The test of a gate

One question decides whether you have built an approval gate or a notification:
**at the moment the human says no, has anything irreversible already happened?**

In [ ]:
def run_gate_before_write(ref: str, approved_by=None) -> dict:
    """Gate placed after the evidence and before the release."""
    checkpoint = run_until_interrupt(ref)
    return resume(checkpoint, approved_by=approved_by)


def run_gate_after_write(ref: str, approved_by=None) -> dict:
    """Gate placed at the end -- the shape that feels thorough and controls nothing."""
    state = {"ref": ref, "tokens": 0}
    for node in PLAN:                       # everything, release included
        state = merge_one(state, NODES[node](state))
    return {"status": "completed" if approved_by else "reviewer said no",
            "state": state, "trace": list(PLAN), "reason": ""}


def no_is_free(run_fn, ref: str) -> bool:
    """Run it, have the human refuse, and ask whether anything happened anyway."""
    RELEASED.clear()
    run_fn(ref, approved_by=None)
    return ref not in RELEASED

In [ ]:
# --- Self-check: Section 4
check("with the gate before the write, saying no costs nothing",
      lambda: no_is_free(run_gate_before_write, "PMT-1005") is True)
check("with the gate after the write, the payment already went",
      lambda: no_is_free(run_gate_after_write, "PMT-1005") is False,
      "the reviewer sees a complete, sourced summary of something they can no longer stop")
check("both runs show the reviewer exactly the same evidence",
      lambda: len(run_gate_after_write("PMT-1005")["state"]["findings"])
              == len(run_until_interrupt("PMT-1005")["state"]["findings"]),
      "quality of evidence was never the difference -- placement was")
check("only the first one is an approval gate",
      lambda: no_is_free(run_gate_before_write, "PMT-1005")
              and not no_is_free(run_gate_after_write, "PMT-1005"))

def _placement():
    for label, fn in (("before the write", run_gate_before_write),
                      ("after the write ", run_gate_after_write)):
        free = no_is_free(fn, "PMT-1005")
        print(f"  gate {label}:  'no' still free? {'yes -- a gate' if free else 'NO -- a notification'}")
guard(_placement)

## Run it for real

Render the checkpoint the way a human reviewer would see it and ask the model to write the
approval request. What you are judging is whether the state you checkpointed contains enough for
a person to say no.

In [ ]:
if llm_ready():
    def _brief():
        cp = run_until_interrupt("PMT-1005")
        evidence = "\n".join(f"- [{f['by']}, source={f['source']}] {f['claim']}"
                             for f in cp["state"]["findings"])
        reply = ask("Write a short approval request for a duty manager. State what is being asked, "
                    "the evidence for and against, and what happens if they do nothing.\n\n"
                    f"Action awaiting approval: {cp['at']} {cp['state']['ref']}\n"
                    f"Agent recommendation: {cp['state'].get('recommendation')}\n"
                    f"Findings:\n{evidence}")
        print(reply.strip()[:600])
    guard(_brief)

### Read it

If the model has to hedge or invent, your checkpoint is missing something a reviewer needs &mdash;
and that is a state design problem, not a prompt problem. A good approval request is mostly a
rendering of state you already had.

**What you take from this lab:** interrupt before the node, not after it; record an identity
rather than a boolean; give every gate a deadline and a ladder that ends; and test placement with
one question &mdash; is &ldquo;no&rdquo; still free?

In [ ]:
score()

## Your turn

1. `resume` trusts its caller for `approved_by`. Where does that name actually have to come from
   for the audit trail to mean anything, and what stops an agent from supplying it?
2. Add a fourth outcome to the gate: *approved with a change* &mdash; the human edits the
   recommendation before resuming. What does that do to the trace, and to who is responsible?
3. `run_until_interrupt` restarts from the checkpoint's remaining nodes. Make the human wait long
   enough that the ledger has changed underneath them, and decide what a resume owes a reviewer
   whose evidence is now stale.